In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from sklearn.linear_model import LinearRegression

# ── Dane ──────────────────────────────────────────────────────────────────────
df = pd.read_csv('daily_ice_edge.csv', parse_dates=['Date'])
LON_COLS = [f'longitude_{i}E' for i in range(360)]

# Kąty (radiany) — zamknięty kontur (361 punktów)
θ = np.deg2rad(np.arange(361))           # 0–360°, ostatni = pierwszy


# ── Funkcja pomocnicza: zamknięcie konturu ────────────────────────────────────
def close(arr):
    """Dołącza pierwszy element na koniec, żeby linia się zamknęła."""
    return np.append(arr, arr[0])


# ── Ustawienia wykresu polarnego ──────────────────────────────────────────────
def polar_ax(fig):
    ax = fig.add_subplot(111, projection='polar')
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_ylim(-90, -50)

    # Ticki szerokości geograficznej
    ax.set_yticks(np.arange(-90, -49, 10))
    ax.set_yticklabels([f'{abs(v)}°S' for v in np.arange(-90, -49, 10)])

    # Ticki azymutu — set_xticks w radianach, bez duplikatów
    ax.set_xticks(np.deg2rad(np.arange(0, 360, 45)))
    ax.set_xticklabels(['0°', '45°', '90°', '135°', '180°', '225°', '270°', '315°'])

    return ax

In [2]:
# ── ETAP 0: minimalny zasięg lodu ────────────────────────────────────────────
print("Etap 0: minimalny zasięg...")
min_edge = df[LON_COLS].min().values

fig = plt.figure(figsize=(10, 10))
ax = polar_ax(fig)
ax.plot(θ, close(min_edge), color='darkblue', lw=2, label='Min. zasięg lodu')
ax.fill_between(θ, -90, close(min_edge), color='lightblue', alpha=0.6)
ax.set_title('Minimalny zasięg lodu morskiego', fontsize=16, pad=20)
ax.legend(loc='lower right', bbox_to_anchor=(1.15, 0))
plt.savefig('minimal_ice_extent.png', bbox_inches='tight')
plt.close()

Etap 0: minimalny zasięg...


In [3]:
# ── ETAP 1: model Fouriera dla każdego południka ─────────────────────────────
print("Etap 1: modelowanie...")
t = (df['Date'] - df['Date'].min()).dt.days.values
w = 2 * np.pi / 365.25
X = np.column_stack([t, np.sin(w * t), np.cos(w * t)])  # trend + sezonowość

params, modeled = [], {'Date': df['Date']}

for i, col in enumerate(LON_COLS):
    y = df[col].values
    mask = ~np.isnan(y)

    if mask.any():
        reg = LinearRegression().fit(X[mask], y[mask])
        a0, (a1, a2, a3) = reg.intercept_, reg.coef_
        params.append({'Angle': i, 'a0': a0, 'a1_trend': a1, 'a2_sin': a2, 'a3_cos': a3})
        modeled[col] = reg.predict(X)
    else:
        params.append({'Angle': i, 'a0': np.nan, 'a1_trend': np.nan, 'a2_sin': np.nan, 'a3_cos': np.nan})
        modeled[col] = np.nan

pd.DataFrame(params).to_csv('ice_model_parameters.csv', index=False)
modeled_df = pd.DataFrame(modeled)
modeled_df.to_csv('modeled_daily_ice_edge.csv', index=False)

Etap 1: modelowanie...


In [4]:
print("Etap 2: globalny model nieliniowy ")

from sklearn.ensemble import HistGradientBoostingRegressor

def make_global_features(lon, t_days, K=8):
    lon = np.asarray(lon)
    t_days = np.asarray(t_days)

    data = {}

    data['t_days'] = t_days
    data['t_sin'] = np.sin(2 * np.pi * t_days / 365.25)
    data['t_cos'] = np.cos(2 * np.pi * t_days / 365.25)

    for k in range(1, K + 1):
        data[f'lon_sin_{k}'] = np.sin(k * np.radians(lon))
        data[f'lon_cos_{k}'] = np.cos(k * np.radians(lon))

        data[f'lon_sin_{k}_t_sin'] = data[f'lon_sin_{k}'] * data['t_sin']
        data[f'lon_sin_{k}_t_cos'] = data[f'lon_sin_{k}'] * data['t_cos']
        data[f'lon_cos_{k}_t_sin'] = data[f'lon_cos_{k}'] * data['t_sin']
        data[f'lon_cos_{k}_t_cos'] = data[f'lon_cos_{k}'] * data['t_cos']

    return pd.DataFrame(data)

df_melted = df.melt(
    id_vars=['Date'],
    value_vars=LON_COLS,
    var_name='Lon_str',
    value_name='Lat'
)

df_melted = df_melted.dropna(subset=['Lat']).copy()

df_melted['Lon'] = df_melted['Lon_str'].str.extract(r'(\d+)').astype(float)

t_min = df['Date'].min()
df_melted['t_days'] = (df_melted['Date'] - t_min).dt.days

X_global = make_global_features(df_melted['Lon'], df_melted['t_days'], K=8)
y_global = df_melted['Lat']

model_global = HistGradientBoostingRegressor(
    max_iter=400,
    learning_rate=0.05,
    max_leaf_nodes=31,
    l2_regularization=0.01,
    random_state=42
)

model_global.fit(X_global, y_global)

print("R2 modelu globalnego:", model_global.score(X_global, y_global))

Etap 2: globalny model nieliniowy 
R2 modelu globalnego: 0.9345837399026573


In [ ]:
print("Etap 3: animacja dla obu metod")

real = df.iloc[::10].reset_index(drop=True)
model_local = modeled_df.iloc[::10].reset_index(drop=True)

global_animated_rows = []
test_lon = np.arange(360)

for i, row in real.iterrows():
    sample_t_days = np.full(360, (row['Date'] - t_min).days)
    X_frame = make_global_features(test_lon, sample_t_days, K=8)
    global_animated_rows.append(model_global.predict(X_frame))

model_global_df = pd.DataFrame(global_animated_rows, columns=LON_COLS)

print(f"Zakres animacji: {real['Date'].iloc[0]} → {real['Date'].iloc[-1]}, klatek: {len(real)}")

fig = plt.figure(figsize=(10, 10))
ax = polar_ax(fig)

line_r, = ax.plot([], [], color='darkblue', lw=2, label='Dane rzeczywiste')
line_l, = ax.plot([], [], color='red', linestyle='--', lw=2, label='Model lokalny Fouriera')
line_g, = ax.plot([], [], color='green', lw=2, label='Model globalny')

title = ax.set_title('', fontsize=16, pad=20)
ax.legend(loc='lower right', bbox_to_anchor=(1.15, 0))

def update(i):
    y_r = close(real.iloc[i][LON_COLS].values.astype(float))
    y_l = close(model_local.iloc[i][LON_COLS].values.astype(float))
    y_g = close(model_global_df.iloc[i][LON_COLS].values.astype(float))

    line_r.set_data(θ, y_r)
    line_l.set_data(θ, y_l)
    line_g.set_data(θ, y_g)

    title.set_text(f"Zasięg lodu: {real.iloc[i]['Date'].strftime('%Y-%m-%d')}")
    return line_r, line_l, line_g, title

ani = animation.FuncAnimation(
    fig,
    update,
    frames=len(real),
    blit=True,
    interval=100
)

ani.save('ice_animation_both_methods.gif', writer='pillow', fps=16)
plt.close()

Etap 3: animacja dla obu metod
Zakres animacji: 1978-10-26 00:00:00 → 2009-05-07 00:00:00, klatek: 953


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

true_long = df.melt(
    id_vars=['Date'],
    value_vars=LON_COLS,
    var_name='Lon_str',
    value_name='Lat_true'
)

local_long = modeled_df.melt(
    id_vars=['Date'],
    value_vars=LON_COLS,
    var_name='Lon_str',
    value_name='Lat_local'
)

compare_df = true_long.merge(local_long, on=['Date', 'Lon_str'])
compare_df = compare_df.dropna(subset=['Lat_true', 'Lat_local']).copy()

compare_df['Lon'] = compare_df['Lon_str'].str.extract(r'(\d+)').astype(float)
compare_df['t_days'] = (compare_df['Date'] - t_min).dt.days

X_global_compare = make_global_features(compare_df['Lon'], compare_df['t_days'], K=8)
compare_df['Lat_global'] = model_global.predict(X_global_compare)

mae_local = mean_absolute_error(compare_df['Lat_true'], compare_df['Lat_local'])
rmse_local = np.sqrt(mean_squared_error(compare_df['Lat_true'], compare_df['Lat_local']))
r2_local = r2_score(compare_df['Lat_true'], compare_df['Lat_local'])

mae_global = mean_absolute_error(compare_df['Lat_true'], compare_df['Lat_global'])
rmse_global = np.sqrt(mean_squared_error(compare_df['Lat_true'], compare_df['Lat_global']))
r2_global = r2_score(compare_df['Lat_true'], compare_df['Lat_global'])

metrics_df = pd.DataFrame({
    'Metoda': ['Model lokalny Fouriera', 'Model globalny'],
    'MAE': [mae_local, mae_global],
    'RMSE': [rmse_local, rmse_global],
    'R²': [r2_local, r2_global]
})

metrics_df_rounded = metrics_df.copy()
metrics_df_rounded[['MAE', 'RMSE', 'R²']] = metrics_df_rounded[['MAE', 'RMSE', 'R²']].round(4)

print(metrics_df_rounded)

metrics_df_rounded.to_csv('comparison_metrics.csv', index=False)